# 面试问题：At-Least-Once Job Queue 怎样用幂等状态机避免重复副作用？

        ## 可直接复述的回答主线

        1. At-least-once 保证消息不会轻易丢失，但 worker 崩溃或 ACK 丢失会让同一 job 再次投递。
2. 业务幂等键必须来自稳定 job_id，而不能用每次 delivery_id，否则重投无法识别。
3. Worker 先以条件写创建 PROCESSING 记录，执行带同一幂等键的副作用，保存 COMPLETED 结果后再 ACK。
4. 收到重复投递时直接返回已完成结果；崩溃发生在副作用之前时允许租约过期后重试。
5. 评测应展示每次 delivery、attempt、故障点、状态迁移和实际副作用次数。
6. 生产还需要事务 inbox/outbox、租约、心跳、DLQ、最大尝试、结果保留期、对账和 poison message 隔离。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例包含六个业务 job、九次实际投递：发票生成与扣款都因 ACK 丢失而重投，缩略图任务在副作用前崩溃后重试，其余为邮件、CRM 同步和导出。数据是离线事件，副作用仅写入内存账本。

In [1]:
import hashlib  # 为业务结果生成确定性摘要。
deliveries = [{"delivery_id": "d-001", "job_id": "invoice-17", "attempt": 1, "kind": "generate_invoice", "payload": "order-17", "failure": "ack_lost_after_effect"}, {"delivery_id": "d-002", "job_id": "invoice-17", "attempt": 2, "kind": "generate_invoice", "payload": "order-17", "failure": None}, {"delivery_id": "d-003", "job_id": "charge-22", "attempt": 1, "kind": "charge_card", "payload": "order-22:88.00", "failure": "ack_lost_after_effect"}, {"delivery_id": "d-004", "job_id": "charge-22", "attempt": 2, "kind": "charge_card", "payload": "order-22:88.00", "failure": None}, {"delivery_id": "d-005", "job_id": "email-31", "attempt": 1, "kind": "send_email", "payload": "receipt-31", "failure": None}, {"delivery_id": "d-006", "job_id": "thumb-08", "attempt": 1, "kind": "render_thumbnail", "payload": "image-08", "failure": "crash_before_effect"}, {"delivery_id": "d-007", "job_id": "thumb-08", "attempt": 2, "kind": "render_thumbnail", "payload": "image-08", "failure": None}, {"delivery_id": "d-008", "job_id": "crm-14", "attempt": 1, "kind": "sync_crm", "payload": "customer-14", "failure": None}, {"delivery_id": "d-009", "job_id": "export-05", "attempt": 1, "kind": "export_report", "payload": "report-05", "failure": None}]  # 定义六个业务 job 的九次 at-least-once 投递。
unique_jobs = sorted({delivery["job_id"] for delivery in deliveries})  # 取得应恰好执行一次的六个稳定业务键。
print("教学实验输入：At-Least-Once 投递日志")  # 标记下方为可复现故障事件。
print("delivery  job          attempt  kind                 failure")  # 输出投递预览表头。
for delivery in deliveries:  # 逐条展示业务键、尝试和故障点。
    print(f"{delivery['delivery_id']:<9} {delivery['job_id']:<12} {delivery['attempt']:>7}  {delivery['kind']:<20} {delivery['failure']}")  # 输出当前消息的真实字段。
print("唯一业务job=", unique_jobs)  # 展示预期副作用基数而不是只看投递数。

教学实验输入：At-Least-Once 投递日志
delivery  job          attempt  kind                 failure
d-001     invoice-17         1  generate_invoice     ack_lost_after_effect
d-002     invoice-17         2  generate_invoice     None
d-003     charge-22          1  charge_card          ack_lost_after_effect
d-004     charge-22          2  charge_card          None
d-005     email-31           1  send_email           None
d-006     thumb-08           1  render_thumbnail     crash_before_effect
d-007     thumb-08           2  render_thumbnail     None
d-008     crm-14             1  sync_crm             None
d-009     export-05          1  export_report        None
唯一业务job= ['charge-22', 'crm-14', 'email-31', 'export-05', 'invoice-17', 'thumb-08']


## 2. Baseline / 基线：每次 delivery 都重新执行

基线把 `delivery_id` 当执行单元。ACK 丢失后 Broker 重投，第二个 delivery_id 会再次生成发票或扣款，产生两个真实副作用。

In [2]:
def effect_result(delivery):  # 为一次业务副作用生成可读确定性结果。
    raw = f"{delivery['kind']}|{delivery['payload']}"  # 构造副作用业务内容。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:12]  # 返回可用于对账的结果摘要。
baseline_effects = []  # 保存基线真正执行的每次副作用。
baseline_ledger = []  # 保存每次 delivery 的执行与 ACK 结果。
for delivery in deliveries:  # 按 Broker 投递顺序处理九条消息。
    if delivery["failure"] == "crash_before_effect":  # 模拟 worker 在调用外部系统前崩溃。
        baseline_ledger.append({"delivery": delivery["delivery_id"], "job": delivery["job_id"], "event": "crash_before_effect", "acked": False})  # 记录未执行且未 ACK。
        continue  # 等待 Broker 后续重投。
    result = effect_result(delivery)  # 对当前 delivery 无条件执行副作用。
    baseline_effects.append({"job": delivery["job_id"], "delivery": delivery["delivery_id"], "result": result})  # 写入外部效果账本。
    acked = delivery["failure"] != "ack_lost_after_effect"  # ACK 丢失时 Broker 不知道副作用已完成。
    baseline_ledger.append({"delivery": delivery["delivery_id"], "job": delivery["job_id"], "event": "effect_executed", "result": result, "acked": acked})  # 保存执行和 ACK 状态。
baseline_counts = {job_id: sum(effect["job"] == job_id for effect in baseline_effects) for job_id in unique_jobs}  # 统计每个业务 job 的实际副作用次数。
baseline_duplicate_effects = sum(max(count - 1, 0) for count in baseline_counts.values())  # 汇总超过一次的重复副作用。
print("Baseline delivery执行账本")  # 标记下表展示 ACK 丢失后的重做。
for event in baseline_ledger:  # 逐事件打印执行或崩溃。
    print(event)  # 输出当前 delivery 的真实结果。
print("Baseline每个job副作用次数=", baseline_counts, "重复次数=", baseline_duplicate_effects)  # 量化发票和扣款重复。

Baseline delivery执行账本
{'delivery': 'd-001', 'job': 'invoice-17', 'event': 'effect_executed', 'result': '9b928bf0b510', 'acked': False}
{'delivery': 'd-002', 'job': 'invoice-17', 'event': 'effect_executed', 'result': '9b928bf0b510', 'acked': True}
{'delivery': 'd-003', 'job': 'charge-22', 'event': 'effect_executed', 'result': '634fe2656cde', 'acked': False}
{'delivery': 'd-004', 'job': 'charge-22', 'event': 'effect_executed', 'result': '634fe2656cde', 'acked': True}
{'delivery': 'd-005', 'job': 'email-31', 'event': 'effect_executed', 'result': '0005a9aa307c', 'acked': True}
{'delivery': 'd-006', 'job': 'thumb-08', 'event': 'crash_before_effect', 'acked': False}
{'delivery': 'd-007', 'job': 'thumb-08', 'event': 'effect_executed', 'result': 'd62376e66fc6', 'acked': True}
{'delivery': 'd-008', 'job': 'crm-14', 'event': 'effect_executed', 'result': '78cef5368f83', 'acked': True}
{'delivery': 'd-009', 'job': 'export-05', 'event': 'effect_executed', 'result': '1412bcebcccd', 'acked': True}
Ba

## 3. 底层实现：稳定幂等键、PROCESSING/COMPLETED 与结果回放

Worker 用 job_id 作为键。副作用提供方也按同一键去重；完成记录先于 ACK 持久化，因此 ACK 丢失后的重投只回放结果，不再次调用外部系统。

In [3]:
idempotency_records = {}  # 保存 job_id 到状态、结果和 owner delivery 的映射。
provider_effects = {}  # 模拟支持业务幂等键的外部副作用系统。
corrected_ledger = []  # 保存每次投递的状态迁移和 ACK 结果。
def call_provider_once(delivery):  # 用稳定 job_id 调用具有幂等语义的外部系统。
    job_id = delivery["job_id"]  # 读取业务幂等键。
    if job_id in provider_effects:  # 检查外部系统是否已完成同一业务操作。
        return provider_effects[job_id], True  # 返回既有结果并标记 provider replay。
    result = effect_result(delivery)  # 首次调用时执行确定性教学副作用。
    provider_effects[job_id] = result  # 按 job_id 保存唯一外部结果。
    return result, False  # 返回新结果并标记真实执行。
for delivery in deliveries:  # 按同一九次投递运行幂等 worker。
    job_id = delivery["job_id"]  # 读取稳定业务键而不是 delivery_id。
    existing = idempotency_records.get(job_id)  # 查询本地 inbox 状态。
    if existing is not None and existing["status"] == "COMPLETED":  # 检查该业务 job 是否已经提交完成。
        corrected_ledger.append({"delivery": delivery["delivery_id"], "job": job_id, "event": "deduplicated", "state_before": "COMPLETED", "result": existing["result"], "provider_called": False, "acked": True})  # 回放结果并 ACK 重投。
        continue  # 不再次调用副作用系统。
    idempotency_records[job_id] = {"status": "PROCESSING", "owner": delivery["delivery_id"], "result": None}  # 以条件写语义取得当前 job 执行权。
    if delivery["failure"] == "crash_before_effect":  # 模拟在副作用前崩溃。
        corrected_ledger.append({"delivery": delivery["delivery_id"], "job": job_id, "event": "lease_abandoned", "state_before": "NONE", "provider_called": False, "acked": False})  # 记录未产生外部效果且未 ACK。
        del idempotency_records[job_id]  # 教学中立即模拟租约过期以允许重投取得执行权。
        continue  # 等待下一次 delivery。
    result, replayed_by_provider = call_provider_once(delivery)  # 用稳定 job_id 执行或回放副作用。
    idempotency_records[job_id] = {"status": "COMPLETED", "owner": delivery["delivery_id"], "result": result}  # 在 ACK 前提交完成记录和结果。
    acked = delivery["failure"] != "ack_lost_after_effect"  # 模拟响应 ACK 是否到达 Broker。
    corrected_ledger.append({"delivery": delivery["delivery_id"], "job": job_id, "event": "completed", "state_before": "NONE", "result": result, "provider_called": not replayed_by_provider, "acked": acked})  # 保存完成、provider 调用和 ACK 状态。
print("幂等 Worker 状态迁移账本")  # 标记下表展示 PROCESSING、COMPLETED 和 deduplicated。
for event in corrected_ledger:  # 逐投递展示核心中间状态。
    print(event)  # 输出当前 delivery 的执行权、结果和 ACK。
print("最终幂等记录=", idempotency_records)  # 展示六个 job 的完成状态与可回放结果。

幂等 Worker 状态迁移账本
{'delivery': 'd-001', 'job': 'invoice-17', 'event': 'completed', 'state_before': 'NONE', 'result': '9b928bf0b510', 'provider_called': True, 'acked': False}
{'delivery': 'd-002', 'job': 'invoice-17', 'event': 'deduplicated', 'state_before': 'COMPLETED', 'result': '9b928bf0b510', 'provider_called': False, 'acked': True}
{'delivery': 'd-003', 'job': 'charge-22', 'event': 'completed', 'state_before': 'NONE', 'result': '634fe2656cde', 'provider_called': True, 'acked': False}
{'delivery': 'd-004', 'job': 'charge-22', 'event': 'deduplicated', 'state_before': 'COMPLETED', 'result': '634fe2656cde', 'provider_called': False, 'acked': True}
{'delivery': 'd-005', 'job': 'email-31', 'event': 'completed', 'state_before': 'NONE', 'result': '0005a9aa307c', 'provider_called': True, 'acked': True}
{'delivery': 'd-006', 'job': 'thumb-08', 'event': 'lease_abandoned', 'state_before': 'NONE', 'provider_called': False, 'acked': False}
{'delivery': 'd-007', 'job': 'thumb-08', 'event': 'comple

## 4. 逐 Job 结果与结果解读

对每个稳定 job_id 比较 Baseline 与幂等 Worker 的真实副作用次数。ACK 可以丢失，但外部业务效果仍应恰好一次。

In [4]:
corrected_counts = {job_id: int(job_id in provider_effects) for job_id in unique_jobs}  # 统计外部 provider 中每个业务键的唯一效果。
corrected_duplicate_effects = sum(max(count - 1, 0) for count in corrected_counts.values())  # 计算修正后的重复效果数。
completed_jobs = sum(record["status"] == "COMPLETED" for record in idempotency_records.values())  # 统计最终完成业务数。
print("job          baseline_effects  idempotent_effects  final_state   result")  # 输出逐 Job 同数据对照表头。
for job_id in unique_jobs:  # 逐业务键比较副作用基数。
    record = idempotency_records[job_id]  # 读取最终完成记录。
    print(f"{job_id:<12} {baseline_counts[job_id]:>16} {corrected_counts[job_id]:>19}  {record['status']:<11} {record['result']}")  # 输出当前 job 的重复情况和结果摘要。
print(f"结果解读：Baseline产生{baseline_duplicate_effects}次重复副作用；幂等状态机完成{completed_jobs}/{len(unique_jobs)}个job，重复副作用={corrected_duplicate_effects}。")  # 解释 at-least-once 与业务 exactly-once effect 的区别。

job          baseline_effects  idempotent_effects  final_state   result
charge-22                   2                   1  COMPLETED   634fe2656cde
crm-14                      1                   1  COMPLETED   78cef5368f83
email-31                    1                   1  COMPLETED   0005a9aa307c
export-05                   1                   1  COMPLETED   1412bcebcccd
invoice-17                  2                   1  COMPLETED   9b928bf0b510
thumb-08                    1                   1  COMPLETED   d62376e66fc6
结果解读：Baseline产生2次重复副作用；幂等状态机完成6/6个job，重复副作用=0。


## 5. 失败案例与修正：并发 check-then-act 竞态

两个 worker 同时读取“记录不存在”后都执行，普通 `if key not in dict` 不是原子门禁。条件插入只允许一个 owner；外部 provider 再以 job_id 去重形成第二道防线。

In [5]:
race_job = "charge-race-99"  # 构造高风险扣款并发重投业务键。
worker_a_saw_absent = True  # 模拟 worker A 在同一快照中看到键不存在。
worker_b_saw_absent = True  # 模拟 worker B 在同一快照中也看到键不存在。
naive_race_effects = int(worker_a_saw_absent) + int(worker_b_saw_absent)  # 两个 worker 都越过非原子检查并执行扣款。
reservation = {}  # 创建支持条件插入语义的教学 inbox。
def compare_and_set_processing(store, job_id, owner):  # 手写只有空键才能取得执行权的条件写。
    if job_id in store:  # 检查是否已有 owner 在同一原子临界区占用。
        return False  # 后到 worker 获取执行权失败。
    store[job_id] = {"status": "PROCESSING", "owner": owner}  # 首个 worker 原子创建租约记录。
    return True  # 返回取得执行权成功。
worker_a_reserved = compare_and_set_processing(reservation, race_job, "worker-a")  # worker A 尝试取得执行权。
worker_b_reserved = compare_and_set_processing(reservation, race_job, "worker-b")  # worker B 紧接着尝试同一键。
corrected_race_effects = int(worker_a_reserved) + int(worker_b_reserved)  # 只有成功 owner 可以调用副作用系统。
print(f"错误行为：A/B同时读到不存在={worker_a_saw_absent}/{worker_b_saw_absent}，实际副作用次数={naive_race_effects}")  # 展示 check-then-act 重复扣款。
print(f"修正行为：A_reserved={worker_a_reserved}，B_reserved={worker_b_reserved}，owner={reservation[race_job]['owner']}，副作用次数={corrected_race_effects}")  # 展示条件插入消除竞态。

错误行为：A/B同时读到不存在=True/True，实际副作用次数=2
修正行为：A_reserved=True，B_reserved=False，owner=worker-a，副作用次数=1


## 6. 生产边界

内存字典不能跨进程原子。生产应使用数据库唯一键与事务 inbox/outbox、带 fencing token 的租约、心跳与超时接管、外部 API idempotency-key、最大尝试、指数退避、DLQ、结果保留/清理和财务对账补偿。

In [6]:
queue_diagnostics = {"deliveries": len(deliveries), "unique_jobs": len(unique_jobs), "baseline_duplicate_effects": baseline_duplicate_effects, "corrected_duplicate_effects": corrected_duplicate_effects, "deduplicated_deliveries": sum(event["event"] == "deduplicated" for event in corrected_ledger), "completed_jobs": completed_jobs, "conditional_write_is_distributed": False}  # 汇总投递、去重和实现边界。
print("生产监控快照：", queue_diagnostics)  # 输出 at-least-once Worker 应持续观察的指标。

生产监控快照： {'deliveries': 9, 'unique_jobs': 6, 'baseline_duplicate_effects': 2, 'corrected_duplicate_effects': 0, 'deduplicated_deliveries': 2, 'completed_jobs': 6, 'conditional_write_is_distributed': False}


## 7. 最小回归测试

断言覆盖业务规模、重复复现、完成结果、ACK 丢失回放和并发竞态修正。

In [7]:
assert len(unique_jobs) >= 6 and len(deliveries) > len(unique_jobs)  # 保证案例至少六个业务 job 且含真实重投。
assert baseline_duplicate_effects == 2 and baseline_counts["charge-22"] == 2  # 保证 ACK 丢失导致重复扣款真实复现。
assert corrected_duplicate_effects == 0 and completed_jobs == len(unique_jobs)  # 保证幂等 Worker 最终每个业务恰好一个效果。
assert sum(event["event"] == "deduplicated" for event in corrected_ledger) == 2  # 保证两次 ACK 丢失重投都走结果回放。
assert all(record["status"] == "COMPLETED" and record["result"] for record in idempotency_records.values())  # 保证完成状态包含可复用结果。
assert naive_race_effects == 2 and corrected_race_effects == 1 and worker_a_reserved != worker_b_reserved  # 保证并发 check-then-act 失败和条件写修正均发生。